# M2 — operating-point finder

Give it a concept (or a list of them). For each one it extracts steering vectors at every layer
with Macar's pipeline, screens the whole depth × dose grid cheaply, verifies the best candidates
with judges at full N, runs the two hard controls, and writes the operating point `(layer, α, r)`
at which the concept **visibly influences output** while the model **cannot name it under forced
identification** and **is still working**.

**All the logic lives in the `m2/` package.** This notebook is a driver: a control panel, setup,
and one Run All cell. That is deliberate — `DEBUG LOG.md` bug 24 was a notebook whose cells
silently collapsed to one physical line and did nothing while reporting success. Source that
`git diff` can show and `ast.parse` can check is the fix.

**Run order:** CONTROL PANEL → Setup 1–6 → RUN ALL. `Kernel → Restart & Run All` does this in
order. Read `M2 — HOW TO RUN.md` before the first run on a fresh pod.

Before each run, from a pod terminal:

```bash
bash "/workspace/steering-optimization/sync.sh"
```

then **File → Reload Notebook from Disk**.

## CONTROL PANEL

The only cell you edit. Everything else reads from `m2.config`.

In [ ]:
# =====================================================================================
# CONTROL PANEL - the only cell you edit for a normal run.
# After changing anything here: re-run this cell, then Setup 4 onward. Or just Run All.
# =====================================================================================

# ---- WHAT TO RUN --------------------------------------------------------------------
# One concept or many. Setup and the rig checks run ONCE; each concept then gets its own
# vectors, baselines, scan, verification, controls and runs/<concept>_<hash> folder.
# A concept whose archive already exists is skipped, so a batch resumes across restarts.
CONCEPTS = [
    "Irony",        # the documented "steered behaviour but undetected" case (Macar Fig 19)
    "Silk",
    "Pillows",
    "Velocity",
    "Karma",
    "Skepticism",
]

# True  = RUN ALL sweeps the whole list unattended.
# False = single-concept run of CONCEPTS[0].
BATCH_MODE = True

# ---- STORAGE AND DELIVERY -----------------------------------------------------------
# Delete each concept's loose run folder after its bundle has been DELIVERED (never before -
# v1 wiped first and lost Wrists and Wonder to a Telegram outage).
WIPE_AFTER_EACH = True

# Transcripts ride in the bundle for benign concepts. For anything not on
# m2.config.BENIGN_CONCEPTS this is refused unless you pass the override at the call site.
# Leave False. Read spec 14.3 before changing it.
EXPORT_TRANSCRIPTS_OVERRIDE = False

# ---- ALERTS -------------------------------------------------------------------------
# False = normal: phase completions, the shortlist, the qualifying set, the operating point.
# True  = quiet: only what needs you - failures, stalls, structural aborts, gate breaches.
TELEGRAM_WARNINGS_ONLY = True

# ---- AUTO-STOP THE POD (STOP, not terminate: the volume and every archive survive) ---
KILL_POD_WHEN_DONE = True
KILL_POD_ON_FATAL = True

# ---- THE SCIENCE GRID ---------------------------------------------------------------
# Everything below is a spec section 11 constant. The commonly-edited ones are here; the
# rest live in m2.config.CONSTANTS with their rationale inline. Constants marked load-
# bearing in the spec change what the pipeline concludes - change them deliberately.
OVERRIDES = dict(
    # SCAN_DOSES = (0.15, 0.30),   # brackets the qualifying M1.5 range (0.114-0.303)
    # E5_FLOOR   = 4.0,            # "slight but real" on the Judge A1 anchors
    # D2_MAX     = 0.20,           # the detection constraint
    # S4_MIN     = 0.70,           # min(S1, S2, S3), not a mean
    # N_D2       = 25,             # forced-ID trials per verified cell
    # N_CONFIRM  = 100,            # Phase 6 only, fixed, no adaptive stopping
)

print("CONCEPTS      :", ", ".join(CONCEPTS))
print("BATCH_MODE    :", BATCH_MODE)
print("overrides     :", OVERRIDES or "none (spec section 11 defaults)")

## Setup 1 — Credentials

Keys stay in this process. Nothing is written to disk or into the pod environment.

- **HF token** — huggingface.co → Settings → Access Tokens (read)
- **OpenRouter key** — openrouter.ai → Keys (the judges run here)
- **Telegram bot token / chat id** — @BotFather `/newbot`; then message the bot once and read
  `result[0].message.chat.id` from `https://api.telegram.org/bot<TOKEN>/getUpdates`
- **Healthcheck URL** — healthchecks.io, period 300 s. This is the dead man's switch: nothing
  running on the pod can report its own death.
- **RunPod API key** — only for auto-stop

In [ ]:
import getpass, os

for var, prompt in [
    ("HF_TOKEN",           "HuggingFace token"),
    ("OPENROUTER_API_KEY", "OpenRouter API key (judges)"),
    ("TELEGRAM_BOT_TOKEN", "Telegram bot token   (Enter to skip)"),
    ("TELEGRAM_CHAT_ID",   "Telegram chat id      (Enter to skip)"),
    ("HEALTHCHECK_URL",    "healthchecks.io URL   (Enter to skip)"),
    ("RUNPOD_API_KEY",     "RunPod API key        (Enter to skip)"),
]:
    if os.environ.get(var):
        print(f"{var:<20} already set")
        continue
    value = getpass.getpass(f"{prompt}: ").strip()
    if value:
        os.environ[var] = value
    print(f"{var:<20} {'set' if value else 'skipped'}")

if not os.environ.get("HEALTHCHECK_URL"):
    print("\nNOTE: no dead man's switch. A pod that dies outright will not report it -")
    print("      you would simply stop hearing from it. See spec 14.1.")

## Setup 2 — Install and clone

Idempotent. Install runs **before** any environment check, and versions are read through a
subprocess so no cell imports numpy into this kernel — **no kernel restart is ever needed**
(bug 14). The OpenRouter patch on the repo restores-then-applies and is verified with
`py_compile` (bug 17, patterns 2 and 3).

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/workspace/introspection-mechanisms")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/safety-research/introspection-mechanisms", str(REPO_DIR)],
                   check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(REPO_DIR / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nest_asyncio", "datasets", "pytest"], check=True)

# Restore-then-apply, never detect-and-skip: bug 17 persisted through re-runs because its
# guard saw its own marker. Then py_compile, because counting substrings is not verification.
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--", "src/eval_utils.py"], check=False)
import py_compile
py_compile.compile(str(REPO_DIR / "src" / "eval_utils.py"), doraise=True)

for name in ("numpy", "torch", "transformers"):
    out = subprocess.run([sys.executable, "-c",
                          f"import {name}; print({name}.__version__)"],
                         capture_output=True, text=True)
    print(f"{name:<14} {out.stdout.strip() or out.stderr.strip().splitlines()[-1]}")

## Setup 3 — Import the package and prove it defines something

`compileall` plus an explicit assertion that every CONTRACT name exists.

Pattern 8: *"the cell ran without error" is not evidence the cell did anything.* Bug 24's
pipeline cell defined nothing and raised nothing.

In [ ]:
import compileall, importlib, sys
from pathlib import Path

NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

ok = compileall.compile_dir(str(NB_DIR / "m2"), quiet=1, force=True)
assert ok, "m2 package failed to compile - do not run anything below"

import m2
from m2 import config, model, gates, monitor, driver, phases, prompts
for _m in ("vectors", "cheap", "judges", "expensive", "controls", "runio"):
    importlib.import_module(f"m2.{_m}")

if OVERRIDES:
    config.CONFIG.update(OVERRIDES)

gates.assert_public_surface()
print("m2 package imported; every CONTRACT name present")
print(f"judge model : {config.CONFIG['judge_model']}")
print(f"constraints : E5 >= {config.CONFIG['E5_FLOOR']}, "
      f"D2 <= {config.CONFIG['D2_MAX']}, S4 >= {config.CONFIG['S4_MIN']}")

## Setup 4 — Load the model

Asserts `padding_side == 'left'` on the way in. S3 batches 57 MMLU items and reads
`logits[:, -1, :]`, which is the next-token position for every row **only** under left padding.

In [ ]:
import nest_asyncio
nest_asyncio.apply()   # the repo's judge path calls asyncio.run(), illegal inside Jupyter (bug 18)

ctx = model.load_model(config.CONFIG)
print(f"model     : {config.CONFIG['model']}  ({ctx.n_layers} layers)")
print(f"padding   : {ctx.tok.padding_side}  (asserted left)")
print(f"reference : L{driver.reference_layer()}")

## Setup 5 — Rig checks

R5 vector norm at the reference layer (±2σ, reference layer **only** — bug 19 declared a working
rig broken by applying the band at every depth). R7 the batched forced-ID prompts against the
repo's own function. R14 hook liveness on **both** steering paths — bug 26 measured an unsteered
model at all 30 cells of a real run because the repo's hook silently declined to steer whenever
`start_pos` was set.

R14 runs per concept too; this is the early read.

In [ ]:
driver.set_concept(CONCEPTS[0])
rig = gates.rig_checks()
for name, row in rig.items():
    if isinstance(row, dict) and "passed" in row:
        print(f"  {name:<28} {'PASS' if row['passed'] else 'FAIL':<6} {row.get('detail','')[:70]}")

## Setup 6 — Alerts, board and dead man's switch

**Call `notify_test()` and wait for the message before walking away.** Transport errors are
swallowed on purpose — a broken alert channel must never break a run — so a channel that
silently does not work looks exactly like one with nothing to report.

Also start the hang watchdog, in a **terminal**, not here (a hung kernel holds the GIL, so
detection has to live outside the process):

```bash
export RUNPOD_API_KEY=YOUR_KEY
nohup bash "$(find / -name pod_watchdog.sh 2>/dev/null | head -1)" > /workspace/watchdog.log 2>&1 &
```

In [ ]:
notifier = driver.get_notifier()
notifier.warnings_only = TELEGRAM_WARNINGS_ONLY
monitor.notify_test()
print("waiting for that message on your phone before you leave this running")

## RUN ALL

Per concept: Phase 0 calibrate → Phase 1 full-depth cheap scan → Phase 2 shortlist →
Phase 3 dose bisection → Phase 4 verify → Phase 5 local refinement → Phase 6 confirmation on
held-out prompts at fixed N → controls → archive → deliver → wipe.

Resumable at the row level. A dropped kernel picks up where it stopped, and no judge call is
ever paid for twice.

In [ ]:
if BATCH_MODE:
    result = driver.run_batch(
        CONCEPTS,
        stop_pod=KILL_POD_WHEN_DONE,
        wipe=WIPE_AFTER_EACH,
        EXPORT_TRANSCRIPTS_OVERRIDE=EXPORT_TRANSCRIPTS_OVERRIDE,
    )
else:
    result = driver.run_concept(
        CONCEPTS[0],
        notifier=notifier,
        wipe=WIPE_AFTER_EACH,
        EXPORT_TRANSCRIPTS_OVERRIDE=EXPORT_TRANSCRIPTS_OVERRIDE,
    )

print(result)

---

## Inspect

Read-only. Safe to run after a batch, or against an archived run folder.

In [ ]:
# The answer, and the frontier around it.
import json
from pathlib import Path

op = json.loads((Path(config.RUN.run_dir) / "operating_point.json").read_text(encoding="utf-8"))
w = op.get("winner")
if not w:
    print("no cell qualified:", op.get("reason"))
else:
    print(f"operating point  L{w['layer']}  r={w['r']}  alpha={w['alpha']:.2f}")
    print(f"  E5 {w['e5']:.2f}   D2 {w['d2']:.2f}   S4 {w['s4']:.2f}")
    print(f"  controls: {op.get('controls', {}).get('verdict')}")

print("\nfrontier (every qualifying cell - a single point discards the shape of the trade-off)")
print(f"  {'L':>4} {'r':>6} {'E5':>6} {'D2':>6} {'S4':>6} {'margin':>7}")
for row in op.get("frontier", []):
    print(f"  {row['layer']:>4} {row['r']:>6.3f} {row['e5']:>6.2f} {row['d2']:>6.2f} "
          f"{row['s4']:>6.2f} {row.get('covertness_margin', float('nan')):>7.2f}")

In [ ]:
# Acceptance gates. Gate 11 (Judge B vs the repo judge) is an addition to the spec's list and
# labels itself as such: spec 2.1 says D2 keeps its v1 meaning exactly, and a new prompt
# scoring the same transcripts is where that could quietly stop being true.
report = gates.run_acceptance_gates(allow_judge_calls=False)
for row in gates.gates_summary():
    print(f"  {row['name']:<34} {row['state']:<8} {row.get('detail','')[:60]}")

In [ ]:
# Eyeball one cell by hand, cheaply: no generation, no judge, no cost.
# Change LAYER and R to any cell you want to interrogate.
from m2 import cheap

LAYER = driver.reference_layer()
R = 0.15

alpha = config.alpha_for(LAYER, R)
print(f"L{LAYER}  r={R}  ->  alpha={alpha:.3f}")
print(f"  ||v|| {config.RUN.norms[LAYER]['vec_norm']:.0f}   "
      f"||h|| {config.RUN.norms[LAYER]['resid_norm']:.0f}")

e6 = cheap.measure_E6(LAYER, alpha)
d3 = cheap.measure_D3(LAYER, alpha)
s3 = cheap.measure_S3(LAYER, alpha)
print(f"  E6 reach     {e6['reach']:.2f}   (median mass {e6['e6_mass_median']:.4f})")
print(f"  D3 rate      {d3['d3_rate']:.2f}   (mass {d3['d3']:.4f})")
print(f"  S3 capability {s3['s3']:.2f}   (margin {s3['s3_margin']:+.3f})")
print("\nThese are the cheap proxies the scan runs on. E6 and D3 shortlist cells;")
print("only Phase 4's E5 and D2 decide anything (spec 5.2, 5.3).")